# Prediksi Log Return Kurs USD/IDR dengan Sentimen Berita

Alur:
1. Preprocessing berita dan kurs
2. Labelling sentimen: **VADER** dan **Loughran-McDonald (LM)**
3. Cocokkan berita ke hari trading, ringkas per hari, gabung dengan kurs, lalu buat target dan fitur kurs
4. Split data sesuai urutan waktu: **70% train, 15% val, 15% test**
5. Simpan dataset final, lalu export bundle untuk tim (artikel bersih, harga + split terkunci, sentimen baseline, data dictionary)
6. Normalisasi (StandardScaler, **fit di train saja**) dan siapkan semua model dengan **parameter default**
7. Bandingkan **Baseline** (fitur kurs saja) vs **Baseline + NLP** (fitur kurs + fitur sentimen) pakai MAE, MSE, RMSE, DirAcc


# 1. Preprocessing

In [2]:
import re
import hashlib
import html
import json
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from langdetect import detect, DetectorFactory, LangDetectException

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    AdaBoostRegressor,
    GradientBoostingRegressor,
)

SEED = 42
DetectorFactory.seed = SEED
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

# Lokasi data dan rentang tanggal yang dipakai di seluruh notebook
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")

START_DATE = pd.Timestamp("2021-09-01")
END_DATE = pd.Timestamp("2026-09-01 23:59:59")

In [3]:
# Fungsi bantu
# Metrik evaluasi dipakai bersama seluruh tim, makanya ada di src/eval_utils.py
sys.path.insert(0, str(Path("src").resolve()))
from eval_utils import evaluate, METRICS

PRICE_FEATURES = [f"kurs_lag{i}" for i in range(1, 6)] + ["kurs_roll_mean5", "kurs_roll_std5"]


def write_table(df, stem):
    """Simpan sebagai parquet; kalau pyarrow tidak ada, jatuh ke CSV."""
    stem = Path(stem)
    stem.parent.mkdir(parents=True, exist_ok=True)
    try:
        path = stem.with_suffix(".parquet")
        df.to_parquet(path, index=False)
    except ImportError:
        path = stem.with_suffix(".csv")
        df.to_csv(path, index=False, date_format="%Y-%m-%d %H:%M:%S")
        print(f"[peringatan] pyarrow tidak ada, {stem.name} disimpan sebagai CSV")
    return path


def articles_fingerprint(articles):
    """Sidik jari tabel artikel, untuk mendeteksi cache embedding yang sudah basi."""
    urls = articles.sort_values("article_id")["url"].astype(str)
    return hashlib.sha1("\n".join(urls).encode("utf-8")).hexdigest()


In [4]:
news_raw = pd.read_csv(RAW_DIR / "cnbc_with_published.csv")
bi_raw = pd.read_csv(RAW_DIR / "bi-usd-rate.csv")

print("Berita geopolitik :", news_raw.shape)
print("Kurs USD BI       :", bi_raw.shape)
news_raw.head()

Berita geopolitik : (16733, 13)
Kurs USD BI       : (1202, 5)


,keyword_matched,title,url,preview_summary,full_text,date_raw,section,published_raw,published_date,published_time,published_timezone,published_iso,scrape_status
0,geopolitical risk,Global shipping authorities warn of maritime trade breakdown amidgeopoliticalturmoil,https://www.cnbc.com/2026/09/08/global-shipping-iran-war-hormuz-rules.html?&qsearchterm=geopolitical risk,Global maritime authorities have warned that the emergence of “parallel systems” threatens to create a two-tier stru...,"Global maritime authorities have warned that the emergence of ""parallel systems"" threatens to create a two-tier stru...",9/8/2026 2:28:41 PM,Markets,2026-09-08T07:28:41+0000,2026-09-08,07:28 AM,0.0,2026-09-08T07:28:41+0000,ok
1,geopolitical risk,Morning Call Sheet: AI rebound meets risinggeopoliticalrisks,https://www.cnbc.com/video/2026/07/31/morning-call-sheet-ai-rebound-meets-rising-geopolitical-risks.html?&qsearchter...,"Peter Tchir, Head of Macro Strategy at Academy Securities, Thomas Martin, Senior Portfolio Manager at GLOBALT Invest...",NaN,7/31/2026 5:58:53 PM,Morning Call,NaN,NaN,NaN,NaN,NaN,not_found
2,geopolitical risk,"Global markets keep shrugging off shocks. Here’s what could break that streak, according to HSBC",https://www.cnbc.com/2026/09/08/global-markets-shrug-off-shocks-hsbc-sees-what-could-break-the-streak.html?&qsearcht...,"Global markets have shrugged off a barrage of shocks in recent years, but HSBC sees several developments that could ...","In this article Global markets have shrugged off a barrage of shocks in recent years, but HSBC sees several developm...",9/8/2026 11:02:43 AM,World Markets,2026-09-08T04:02:43+0000,2026-09-08,04:02 AM,0.0,2026-09-08T04:02:43+0000,ok
3,geopolitical risk,"Yen hovers near seven-month high, dollar steadies",https://www.cnbc.com/2026/09/08/yen-extends-rally-to-new-seven-month-high-dollar-subdued-ahead-of-cpi.html?&qsearcht...,"The Japanese yen hovered near ​a seven-month high on Tuesday, while the dollar was little changed against major peer...","In this article The Japanese yen hovered near ​a seven-month high on Tuesday, while the dollar was little changed ag...",9/8/2026 11:17:07 AM,Currencies,2026-09-08T04:17:07+0000,2026-09-08,04:17 AM,0.0,2026-09-08T04:17:07+0000,ok
4,geopolitical risk,Central banks are bringing gold reserves home asgeopoliticalrisks rise,https://www.cnbc.com/2026/06/17/central-banks-gold-reserves-domestic-storage.html?&qsearchterm=geopolitical risk,"More and more central banks are storing gold bullion at home rather than overseas, as they expect to buy more of the...","In this article More and more central banks are storing gold bullion at home rather than overseas, as they expect to...",6/17/2026 2:28:36 PM,Gold,2026-06-17T07:28:36+0000,2026-06-17,07:28 AM,0.0,2026-06-17T07:28:36+0000,ok


In [5]:
news_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16733 entries, 0 to 16732
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   keyword_matched     16733 non-null  object 
 1   title               16733 non-null  object 
 2   url                 16733 non-null  object 
 3   preview_summary     16733 non-null  object 
 4   full_text           13431 non-null  object 
 5   date_raw            16733 non-null  object 
 6   section             16649 non-null  object 
 7   published_raw       15239 non-null  object 
 8   published_date      15239 non-null  object 
 9   published_time      15239 non-null  object 
 10  published_timezone  15239 non-null  float64
 11  published_iso       15239 non-null  object 
 12  scrape_status       16733 non-null  object 
dtypes: float64(1), object(12)
memory usage: 1.7+ MB


In [6]:
# Ubah date_raw jadi datetime. Untuk kolom tanggal & jam, date_raw dianggap UTC lalu dikonversi ke WIB.
news_raw["date_raw_dt"] = pd.to_datetime(news_raw["date_raw"], errors="coerce")
news_raw["date_raw_dt_wib"] = (
    pd.to_datetime(news_raw["date_raw_dt"], errors="coerce", utc=True)
    .dt.tz_convert("Asia/Jakarta")
    .dt.tz_localize(None)
)
news_raw["tanggal"] = news_raw["date_raw_dt_wib"].dt.strftime("%m/%d/%Y")
news_raw["jam"] = news_raw["date_raw_dt_wib"].dt.strftime("%I:%M %p")
news_raw = news_raw.drop(columns=["date_raw"], errors="ignore")

print("Kolom Berita Geopolitik:")
news_raw[["title", "tanggal", "jam", "date_raw_dt_wib"]].head()

Kolom Berita Geopolitik:


,title,tanggal,jam,date_raw_dt_wib
0,Global shipping authorities warn of maritime trade breakdown amidgeopoliticalturmoil,09/08/2026,09:28 PM,2026-09-08 21:28:41
1,Morning Call Sheet: AI rebound meets risinggeopoliticalrisks,08/01/2026,12:58 AM,2026-08-01 00:58:53
2,"Global markets keep shrugging off shocks. Here’s what could break that streak, according to HSBC",09/08/2026,06:02 PM,2026-09-08 18:02:43
3,"Yen hovers near seven-month high, dollar steadies",09/08/2026,06:17 PM,2026-09-08 18:17:07
4,Central banks are bringing gold reserves home asgeopoliticalrisks rise,06/17/2026,09:28 PM,2026-06-17 21:28:36


In [7]:
bi_raw["Tanggal_dt"] = pd.to_datetime(bi_raw["Tanggal"], errors="coerce")
bi_raw["tanggal"] = bi_raw["Tanggal_dt"].dt.strftime("%m/%d/%Y")
bi_raw["jam"] = bi_raw["Tanggal_dt"].dt.strftime("%I:%M %p")
bi_raw = bi_raw.drop(columns=["Tanggal"], errors="ignore")

print("Kolom Kurs BI:")
bi_raw.head()

Kolom Kurs BI:


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam
0,1,1,17834.73,17657.27,2026-09-01,09/01/2026,12:00 AM
1,2,1,17791.51,17614.49,2026-08-31,08/31/2026,12:00 AM
2,3,1,17850.81,17673.19,2026-08-28,08/28/2026,12:00 AM
3,4,1,17805.58,17628.42,2026-08-27,08/27/2026,12:00 AM
4,5,1,17791.51,17614.49,2026-08-26,08/26/2026,12:00 AM


## 1.1 Cleaning berita

In [8]:
news = news_raw.copy()
print(f"Baris awal                    : {len(news)}")

# Ambil berita di rentang 1 Sep 2021 - 1 Sep 2026 (dalam WIB, konsisten dengan penyelarasan di bagian 3.1)
news = news[news["date_raw_dt_wib"].between(START_DATE, END_DATE, inclusive="both")]
print(f"Setelah filter rentang tanggal: {len(news)}")

# Buang duplikat: URL yang sama, lalu judul + waktu terbit yang sama
news = news.drop_duplicates(subset=["url"], keep="first")
print(f"Setelah dedup URL             : {len(news)}")

news = news.drop_duplicates(subset=["title", "tanggal", "jam"], keep="first")
print(f"Setelah dedup judul + waktu   : {len(news)}")

Baris awal                    : 16733
Setelah filter rentang tanggal: 8869
Setelah dedup URL             : 8869
Setelah dedup judul + waktu   : 7082


In [9]:
# Buang berita yang judul, ringkasan, dan isinya kosong semua
text_columns = ["title", "preview_summary", "full_text"]
all_text = news[text_columns].fillna("").astype(str).agg(" ".join, axis=1).str.strip()
news = news.loc[all_text.ne("")].copy()
print(f"Setelah drop teks kosong       : {len(news)}")

# Buang konten yang bukan artikel (video, foto, audio, live, dll), dilihat dari URL, judul, dan section
NON_ARTICLE_PATTERN = (
    r"\b(video|videos|image|images|photo|photos|foto|gambar|audio|podcast|gallery|galeri|live)\b"
    r"|\.(mp4|mov|avi|mp3|wav|jpg|jpeg|png)(?:$|[?#])"
)
metadata = news[["url", "title", "section"]].fillna("").astype(str).agg(" ".join, axis=1)
is_non_article = metadata.str.contains(NON_ARTICLE_PATTERN, case=False, regex=True, na=False)
news = news.loc[~is_non_article].copy()
print(f"Setelah drop video/gambar/audio: {len(news)}")

news.head()

Setelah drop teks kosong       : 7082
Setelah drop video/gambar/audio: 6212


,keyword_matched,title,url,preview_summary,full_text,section,published_raw,published_date,published_time,published_timezone,published_iso,scrape_status,date_raw_dt,date_raw_dt_wib,tanggal,jam
4,geopolitical risk,Central banks are bringing gold reserves home asgeopoliticalrisks rise,https://www.cnbc.com/2026/06/17/central-banks-gold-reserves-domestic-storage.html?&qsearchterm=geopolitical risk,"More and more central banks are storing gold bullion at home rather than overseas, as they expect to buy more of the...","In this article More and more central banks are storing gold bullion at home rather than overseas, as they expect to...",Gold,2026-06-17T07:28:36+0000,2026-06-17,07:28 AM,0.0,2026-06-17T07:28:36+0000,ok,2026-06-17 14:28:36,2026-06-17 21:28:36,06/17/2026,09:28 PM
8,geopolitical risk,Oil rises 2% as White House says no US-Iran talks happening,https://www.cnbc.com/2026/08/27/oil-prices-extend-losses-on-expectations-talks-to-ease-middle-east-supply-woes.html?...,"Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomatic efforts by ot...","In this article Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomat...",Oil,2026-08-27T02:49:03+0000,2026-08-27,02:49 AM,0.0,2026-08-27T02:49:03+0000,ok,2026-08-27 09:49:03,2026-08-27 16:49:03,08/27/2026,04:49 PM
9,geopolitical risk,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,https://www.cnbc.com/2026/07/21/jpmorgan-chase-ceo-jamie-dimon-market-risk.html?&qsearchterm=geopolitical risk,JPMorgan Chase CEO Jamie Dimon said investors are underestimating the risks facing the global economy and that he wo...,In this article JPMorgan ChaseCEOJamie Dimonsaid investors are underestimating the risks facing the global economy a...,Finance,2026-07-20T23:01:01+0000,2026-07-20,11:01 PM,0.0,2026-07-20T23:01:01+0000,ok,2026-07-21 06:01:01,2026-07-21 13:01:01,07/21/2026,01:01 PM
13,geopolitical risk,"Citi Wealth warns markets may be ‘uncomfortably strong’ amid mountinggeopolitical, inflation risks",https://www.cnbc.com/2026/05/18/citi-wealth-warns-markets-may-be-uncomfortably-strong-as-risks-mount.html?&qsearchte...,"Global markets may be due for a period of consolidation after a sharp rally in equities, even as the longer-term out...",NaN,Pro: Analysis,2026-05-18T05:37:18+0000,2026-05-18,05:37 AM,0.0,2026-05-18T05:37:18+0000,ok,2026-05-18 12:37:18,2026-05-18 19:37:18,05/18/2026,07:37 PM
15,geopolitical risk,Where fixed income investors are finding yield asgeopoliticalriskrattles markets,https://www.cnbc.com/2026/04/10/where-fixed-income-investors-are-finding-yield-as-geopolitical-risk-rattles-markets....,"As the Iran war shakes up markets, strategists say there are still plenty of sources of relatively safe yield for in...",NaN,Pro: Income Investing,2026-04-10T19:25:17+0000,2026-04-10,07:25 PM,0.0,2026-04-10T19:25:17+0000,ok,2026-04-11 02:25:17,2026-04-11 09:25:17,04/11/2026,09:25 AM


In [10]:
# Ambil artikel yang isinya minimal 350 kata (kalau full_text kosong, pakai preview_summary)
MIN_WORDS = 350


def word_count(text):
    return len(text.split()) if isinstance(text, str) else 0


article_text = news["full_text"].fillna(news["preview_summary"])
news = news.loc[article_text.apply(word_count) >= MIN_WORDS]
print(f"Setelah filter minimal {MIN_WORDS} kata: {len(news)}")

Setelah filter minimal 350 kata: 4347


In [11]:
# Simpan berita yang relevan dengan geopolitik/ekonomi, lalu buang section yang jelas di luar topik
RELEVANCE_KEYWORDS = [
    "geopolitics", "geopolitical risk", "geopolitical tensions", "geopolitical fragmentation",
    "armed conflict", "global conflict", "international conflict", "military tensions",
    "national security", "sanctions", "trade war", "tariffs", "embargo", "export controls",
    "protectionism", "supply chain disruption", "diplomacy", "foreign policy",
    "nato", "brics", "opec", "g7",
]

EXCLUDE_SECTIONS = [
    "sports", "entertainment", "television", "lifestyle", "travel",
    "food retail", "beer, wine & spirits", "college", "modern medicine",
    "life changes", "fashion", "celebrity",
]


def is_relevant(row):
    # Relevan kalau keyword_matched terisi, atau judul/ringkasan memuat salah satu keyword di atas
    if isinstance(row["keyword_matched"], str) and row["keyword_matched"].strip():
        return True
    text = f"{row.get('title', '')} {row.get('preview_summary', '')}".lower()
    return any(kw in text for kw in RELEVANCE_KEYWORDS)


news = news.loc[news.apply(is_relevant, axis=1)]
print(f"Setelah filter keyword: {len(news)}")

section = news["section"].fillna("").str.lower()
news = news.loc[~section.isin(EXCLUDE_SECTIONS)].reset_index(drop=True)
print(f"Setelah filter section: {len(news)}")

news.head(3)

Setelah filter keyword: 4347
Setelah filter section: 4334


,keyword_matched,title,url,preview_summary,full_text,section,published_raw,published_date,published_time,published_timezone,published_iso,scrape_status,date_raw_dt,date_raw_dt_wib,tanggal,jam
0,geopolitical risk,Central banks are bringing gold reserves home asgeopoliticalrisks rise,https://www.cnbc.com/2026/06/17/central-banks-gold-reserves-domestic-storage.html?&qsearchterm=geopolitical risk,"More and more central banks are storing gold bullion at home rather than overseas, as they expect to buy more of the...","In this article More and more central banks are storing gold bullion at home rather than overseas, as they expect to...",Gold,2026-06-17T07:28:36+0000,2026-06-17,07:28 AM,0.0,2026-06-17T07:28:36+0000,ok,2026-06-17 14:28:36,2026-06-17 21:28:36,06/17/2026,09:28 PM
1,geopolitical risk,Oil rises 2% as White House says no US-Iran talks happening,https://www.cnbc.com/2026/08/27/oil-prices-extend-losses-on-expectations-talks-to-ease-middle-east-supply-woes.html?...,"Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomatic efforts by ot...","In this article Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomat...",Oil,2026-08-27T02:49:03+0000,2026-08-27,02:49 AM,0.0,2026-08-27T02:49:03+0000,ok,2026-08-27 09:49:03,2026-08-27 16:49:03,08/27/2026,04:49 PM
2,geopolitical risk,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,https://www.cnbc.com/2026/07/21/jpmorgan-chase-ceo-jamie-dimon-market-risk.html?&qsearchterm=geopolitical risk,JPMorgan Chase CEO Jamie Dimon said investors are underestimating the risks facing the global economy and that he wo...,In this article JPMorgan ChaseCEOJamie Dimonsaid investors are underestimating the risks facing the global economy a...,Finance,2026-07-20T23:01:01+0000,2026-07-20,11:01 PM,0.0,2026-07-20T23:01:01+0000,ok,2026-07-21 06:01:01,2026-07-21 13:01:01,07/21/2026,01:01 PM


In [12]:
# Buang berita yang full_text-nya kosong
print("Jumlah full_text NaN:", news["full_text"].isna().sum())

news = news.dropna(subset=["full_text"]).reset_index(drop=True)
print("Jumlah baris setelah drop NaN:", len(news))

Jumlah full_text NaN: 0
Jumlah baris setelah drop NaN: 4334


In [13]:
# Diagnosis saja (teks tidak diubah): cek judul/ringkasan yang katanya menempel,
# misalnya "risinggeopoliticalrisks". Kemungkinan tag highlight keyword dibuang scraper tanpa spasi.
# Ini memengaruhi skor VADER/LM judul dan embedding, jadi sebaiknya diperbaiki di scraper.
GLUED_RE = re.compile(r"[a-z]geopolitic|geopolitical(?!ly\b)[a-z]", re.IGNORECASE)

for col in ["title", "preview_summary"]:
    glued = news[col].fillna("").str.contains(GLUED_RE)
    print(f"{col:16}: {glued.sum()} dari {len(news)} berisi kata menempel")
    print(news.loc[glued, col].head(3).to_string(index=False), "\n")


title           : 58 dari 4334 berisi kata menempel
                                                 Central banks are bringing gold reserves home asgeopoliticalrisks rise
‘A matter of national survival’: European governments on how they’re accelerating digital sovereignty asgeopolitical...
                          Jamie Dimon saysgeopoliticalrisks are surging: ‘Conditions are treacherous and getting worse’ 

preview_summary : 160 dari 4334 berisi kata menempel
Wall Street’s so-called “fear gauge” has fallen to its lowest level of 2026 so far, reflecting relative market tranqu...
Japan’s largest banks posted record annual profits in their latest financial results, but earnings growth could slow ...
JPMorgan Chase CEO Jamie Dimon is calling for a broad recommitment to American ideals as his bank navigatesgeopolitic... 



In [14]:
# Gabungkan judul, ringkasan, dan isi jadi satu teks (judul ditulis dua kali)
text_columns = ["title", "preview_summary", "full_text"]
news[text_columns] = news[text_columns].fillna("")

news["combined_raw"] = (
    (news["title"] + " ") * 2
    + news["preview_summary"] + " "
    + news["full_text"]
)

print(f"Jumlah baris: {len(news)}")
news[["title", "combined_raw"]].head(3)

Jumlah baris: 4334


,title,combined_raw
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...
1,Oil rises 2% as White House says no US-Iran talks happening,Oil rises 2% as White House says no US-Iran talks happening Oil rises 2% as White House says no US-Iran talks happen...
2,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ... Jamie Dimon says...


In [15]:
# Bersihkan teks: buang HTML, URL, email, emoji, dan tanda baca.
# Simbol % dan mata uang diubah jadi kata (misalnya $ -> dollar). Angka tetap disimpan.
URL_RE = re.compile(r"http\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+\.\S+")
HTML_TAG_RE = re.compile(r"<[^>]+>")
EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001FAFF"
    "\U00002700-\U000027BF"
    "\U0001F1E6-\U0001F1FF"
    "\U00002600-\U000026FF"
    "]+",
    flags=re.UNICODE,
)

SYMBOL_WORDS = {"%": "percent", "$": "dollar", "£": "pound", "€": "euro", "¥": "yen"}
SYMBOL_RE = re.compile("|".join(re.escape(sym) for sym in SYMBOL_WORDS))
NON_ALNUM_RE = re.compile(r"[^a-zA-Z0-9\s]")
MULTI_SPACE_RE = re.compile(r"\s+")


def normalize_text(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    t = html.unescape(text)
    t = HTML_TAG_RE.sub(" ", t)
    t = URL_RE.sub(" ", t)
    t = EMAIL_RE.sub(" ", t)
    t = EMOJI_RE.sub(" ", t)
    t = SYMBOL_RE.sub(lambda m: f" {SYMBOL_WORDS[m.group(0)]} ", t)
    t = NON_ALNUM_RE.sub(" ", t)
    return MULTI_SPACE_RE.sub(" ", t).strip()


news["normalized_text"] = news["combined_raw"].apply(normalize_text)
news[["combined_raw", "normalized_text"]].head(3)

,combined_raw,normalized_text
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...
1,Oil rises 2% as White House says no US-Iran talks happening Oil rises 2% as White House says no US-Iran talks happen...,Oil rises 2 percent as White House says no US Iran talks happening Oil rises 2 percent as White House says no US Ira...
2,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ... Jamie Dimon says...,Jamie Dimon says markets underestimate risks and he wouldn t buy stocks or Treasurys at current Jamie Dimon says mar...


In [16]:
news["lower_text"] = news["normalized_text"].str.lower()
news[["normalized_text", "lower_text"]].head(3)

,normalized_text,lower_text
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...,central banks are bringing gold reserves home asgeopoliticalrisks rise central banks are bringing gold reserves home...
1,Oil rises 2 percent as White House says no US Iran talks happening Oil rises 2 percent as White House says no US Ira...,oil rises 2 percent as white house says no us iran talks happening oil rises 2 percent as white house says no us ira...
2,Jamie Dimon says markets underestimate risks and he wouldn t buy stocks or Treasurys at current Jamie Dimon says mar...,jamie dimon says markets underestimate risks and he wouldn t buy stocks or treasurys at current jamie dimon says mar...


In [17]:
# Pastikan semua berita berbahasa Inggris
def detect_language(text):
    if not isinstance(text, str) or len(text.strip()) < 20:
        return "unknown"
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"


news["detected_lang"] = news["lower_text"].apply(detect_language)
print("Distribusi bahasa:")
print(news["detected_lang"].value_counts().head(10))

n_before = len(news)
news = news.loc[news["detected_lang"] == "en"].reset_index(drop=True)
print(f"\nSetelah filter bahasa Inggris: {len(news)} (dibuang {n_before - len(news)} baris)")

news[["lower_text", "detected_lang"]].head(3)

Distribusi bahasa:
detected_lang
en    4334
Name: count, dtype: int64

Setelah filter bahasa Inggris: 4334 (dibuang 0 baris)


,lower_text,detected_lang
0,central banks are bringing gold reserves home asgeopoliticalrisks rise central banks are bringing gold reserves home...,en
1,oil rises 2 percent as white house says no us iran talks happening oil rises 2 percent as white house says no us ira...,en
2,jamie dimon says markets underestimate risks and he wouldn t buy stocks or treasurys at current jamie dimon says mar...,en


## 1.2 Cleaning data kurs

In [18]:
# Gabungkan kolom tanggal & jam jadi satu kolom datetime
bi_raw["Tanggal"] = pd.to_datetime(
    bi_raw["tanggal"].astype(str).str.strip() + " " + bi_raw["jam"].astype(str).str.strip(),
    format="%m/%d/%Y %I:%M %p",
    errors="coerce",
)

print(bi_raw.shape)
bi_raw.head(3)

(1202, 8)


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam,Tanggal
0,1,1,17834.73,17657.27,2026-09-01,09/01/2026,12:00 AM,2026-09-01
1,2,1,17791.51,17614.49,2026-08-31,08/31/2026,12:00 AM,2026-08-31
2,3,1,17850.81,17673.19,2026-08-28,08/28/2026,12:00 AM,2026-08-28


In [19]:
def clean_numeric(series):
    # Hapus pemisah ribuan (koma), lalu ubah ke angka
    cleaned = series.astype(str).str.replace(",", "", regex=False).str.strip()
    return pd.to_numeric(cleaned, errors="coerce")


bi_raw["Kurs Jual"] = clean_numeric(bi_raw["Kurs Jual"])
bi_raw["Kurs Beli"] = clean_numeric(bi_raw["Kurs Beli"])
bi_raw["NO"] = pd.to_numeric(bi_raw["NO"], errors="coerce")
bi_raw["Nilai"] = pd.to_numeric(bi_raw["Nilai"], errors="coerce")
print(f"Baris awal                : {len(bi_raw)}")

bi = bi_raw.loc[bi_raw["Tanggal"].between(START_DATE, END_DATE, inclusive="both")]
print(f"Setelah filter tanggal    : {len(bi)}")

# Kurs jual & beli harus terisi dan lebih dari 0
is_valid = (
    bi["Kurs Jual"].notna() & bi["Kurs Beli"].notna()
    & (bi["Kurs Jual"] > 0) & (bi["Kurs Beli"] > 0)
)
bi = bi.loc[is_valid]
print(f"Setelah filter kurs valid : {len(bi)}")

# Kurs jual harus lebih tinggi dari kurs beli
bi = bi.loc[bi["Kurs Jual"] > bi["Kurs Beli"]]
print(f"Setelah filter Jual > Beli: {len(bi)}")

# Kalau ada tanggal dobel, simpan satu saja
bi = (
    bi.sort_values("Tanggal")
    .drop_duplicates(subset=["Tanggal"], keep="last")
    .reset_index(drop=True)
)
print(f"Setelah dedup tanggal     : {len(bi)}")

bi.head(3)

Baris awal                : 1202
Setelah filter tanggal    : 1202
Setelah filter kurs valid : 1202
Setelah filter Jual > Beli: 1202
Setelah dedup tanggal     : 1202


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam,Tanggal
0,1202,1,14377.53,14234.47,2021-09-01,09/01/2021,12:00 AM,2021-09-01
1,1201,1,14355.42,14212.58,2021-09-02,09/02/2021,12:00 AM,2021-09-02
2,1200,1,14352.41,14209.60,2021-09-03,09/03/2021,12:00 AM,2021-09-03


# 2. Data Labelling (VADER & LM)

## 2.1 VADER

In [20]:
USE_VADER = True  # ganti ke False kalau VADER tidak mau dipakai

nltk.download("vader_lexicon", quiet=True)
vader = SentimentIntensityAnalyzer()


def vader_score(text):
    # Judul kosong dibiarkan NaN, jangan dianggap netral (0)
    if isinstance(text, str) and text.strip():
        return vader.polarity_scores(text)["compound"]
    return np.nan


news = news.drop(columns=["vader_title"], errors="ignore")  # hapus hasil run sebelumnya

if USE_VADER:
    # Skor dihitung dari judul asli, tanpa cleaning atau lowercase
    news["vader_title"] = news["title"].apply(vader_score)
    print("VADER aktif")
    display(news[["title", "vader_title"]].head())
else:
    print("VADER dinonaktifkan")

VADER aktif


,title,vader_title
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise,0.0000
1,Oil rises 2% as White House says no US-Iran talks happening,-0.2960
2,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,-0.5106
3,China’s super-rich fled Singapore. Now they want to come back,0.0772
4,‘Don’t get too comfortable’: Wall Street’s ‘fear gauge’ hits 2026 low — here’s why it’s ...,-0.2732


## 2.2 Loughran-McDonald

In [21]:
# Kategori LM yang dipakai. Bisa ditambah 'constraining', 'strong_modal', atau 'weak_modal'.
# 'negative' dan 'positive' wajib ada karena dipakai untuk skor net.
LM_CATS = ["negative", "positive", "uncertainty", "litigious"]
assert {"negative", "positive"} <= set(LM_CATS), "LM_CATS harus memuat 'negative' dan 'positive'"

LM_COLS = {
    "negative": "Negative",
    "positive": "Positive",
    "uncertainty": "Uncertainty",
    "litigious": "Litigious",
    "constraining": "Constraining",
    "strong_modal": "Strong_Modal",
    "weak_modal": "Weak_Modal",
}

# Kamus Loughran-McDonald Master Dictionary 1993-2025 (sraf.nd.edu)
LM_FILE = RAW_DIR / "Loughran-McDonald_MasterDictionary_1993-2025.csv"
if not LM_FILE.exists():
    raise FileNotFoundError(f"Kamus LM tidak ditemukan di {LM_FILE}")

lm_df = pd.read_csv(LM_FILE, usecols=["Word"] + [LM_COLS[c] for c in LM_CATS])
lm_df = lm_df.dropna(subset=["Word"])
lm_df["Word"] = lm_df["Word"].astype(str).str.lower()

# Daftar kata untuk tiap kategori
LM_SETS = {c: set(lm_df.loc[lm_df[LM_COLS[c]] > 0, "Word"]) for c in LM_CATS}
print(f"Kamus LM dari: {LM_FILE}")
print({c: len(words) for c, words in LM_SETS.items()})

WORD_RE = re.compile(r"[a-z]+")


def lm_scores(text):
    # Proporsi kata tiap kategori LM dibanding total kata. Teks harus sudah lowercase.
    tokens = WORD_RE.findall(text) if isinstance(text, str) else []
    if not tokens:
        return [np.nan] * len(LM_CATS)  # teks kosong jadi NaN, bukan 0
    return [sum(t in LM_SETS[c] for t in tokens) / len(tokens) for c in LM_CATS]


def lm_columns(texts, prefix):
    # Skor LM tiap artikel + skor net (positive - negative), dengan nama kolom diawali prefix
    scores = texts.apply(lambda t: lm_scores(normalize_text(t).lower())).tolist()
    lm = pd.DataFrame(scores, columns=[f"{prefix}{c}" for c in LM_CATS], index=texts.index)
    lm[f"{prefix}net"] = lm[f"{prefix}positive"] - lm[f"{prefix}negative"]
    return lm

Kamus LM dari: data\raw\Loughran-McDonald_MasterDictionary_1993-2025.csv
{'negative': 2345, 'positive': 347, 'uncertainty': 297, 'litigious': 903}


In [22]:
USE_LM_TITLE = True  # ganti ke False kalau skor LM judul tidak mau dipakai

news = news.drop(columns=[c for c in news.columns if c.startswith("lm_title_")])  # hapus hasil run sebelumnya

if USE_LM_TITLE:
    lm_title = lm_columns(news["title"], prefix="lm_title_")
    news = news.join(lm_title)
    print("LM judul aktif")
    display(news[["title", *lm_title.columns]].head())
else:
    print("LM judul dinonaktifkan")

LM judul aktif


,title,lm_title_negative,lm_title_positive,lm_title_uncertainty,lm_title_litigious,lm_title_net
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise,0.000000,0.0,0.0000,0.0,0.000000
1,Oil rises 2% as White House says no US-Iran talks happening,0.000000,0.0,0.0000,0.0,0.000000
2,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,0.062500,0.0,0.0625,0.0,-0.062500
3,China’s super-rich fled Singapore. Now they want to come back,0.000000,0.0,0.0000,0.0,0.000000
4,‘Don’t get too comfortable’: Wall Street’s ‘fear gauge’ hits 2026 low — here’s why it’s ...,0.058824,0.0,0.0000,0.0,-0.058824


In [23]:
USE_LM_TEXT = True  # ganti ke False kalau skor LM isi berita tidak mau dipakai

news = news.drop(columns=[c for c in news.columns if c.startswith("lm_text_")])  # hapus hasil run sebelumnya

if USE_LM_TEXT:
    # Tidak pakai stopword removal/lemmatisasi, karena kamus LM sudah memuat bentuk turunan kata
    lm_text = lm_columns(news["full_text"], prefix="lm_text_")
    news = news.join(lm_text)
    print("LM isi aktif")
    display(lm_text.describe().T)
else:
    print("LM isi dinonaktifkan")

LM isi aktif


,count,mean,std,min,25%,50%,75%,max
lm_text_negative,4334.0,0.022890,0.010348,0.000000,0.015219,0.022097,0.029221,0.069705
lm_text_positive,4334.0,0.007591,0.005134,0.000000,0.003950,0.006700,0.010243,0.042959
lm_text_uncertainty,4334.0,0.009785,0.006004,0.000000,0.005408,0.008674,0.013126,0.042017
lm_text_litigious,4334.0,0.002633,0.005033,0.000000,0.000000,0.001310,0.002976,0.081146
lm_text_net,4334.0,-0.015299,0.012595,-0.067024,-0.023058,-0.014854,-0.006957,0.041766


# 3. Penyelarasan, Agregasi Harian, Target & Fitur Baseline

## 3.1 Penyelarasan berita ke hari trading

In [24]:
# Berita yang terbit lewat jam 15:00 WIB dihitung ke hari berikutnya.
# Kalau jatuhnya Sabtu/Minggu, digeser ke Senin.
# Pakai waktu WIB (date_raw_dt_wib). date_raw_dt masih UTC, jadi cutoff 15:00 di sana
# sebenarnya 22:00 WIB dan artikel pukul 15:00-22:00 WIB salah dipasang ke hari yang sama.
CUTOFF_HOUR = 15


def align_to_trading_date(dt):
    if dt > dt.replace(hour=CUTOFF_HOUR, minute=0, second=0, microsecond=0):
        dt += pd.Timedelta(days=1)

    if dt.weekday() == 5:    # Sabtu
        dt += pd.Timedelta(days=2)
    elif dt.weekday() == 6:  # Minggu
        dt += pd.Timedelta(days=1)

    return dt.normalize()  # ambil tanggalnya saja


assert news["date_raw_dt_wib"].notna().all(), "ada waktu terbit WIB yang kosong"
news["aligned_trading_date"] = news["date_raw_dt_wib"].apply(align_to_trading_date)

# Pembanding: hasil versi lama (jam UTC). Ini cuma untuk melihat dampak perbaikan.
old_aligned = news["date_raw_dt"].apply(align_to_trading_date)
changed = (old_aligned != news["aligned_trading_date"]).sum()
print(f"Artikel yang tanggal alignment-nya berubah setelah perbaikan timezone: {changed} dari {len(news)}")


Artikel yang tanggal alignment-nya berubah setelah perbaikan timezone: 1426 dari 4334


In [25]:
# Geser lagi ke hari trading terdekat yang benar-benar ada di data kurs (misalnya kalau kena tanggal merah)
trading_days = pd.DatetimeIndex(bi["Tanggal_dt"]).normalize().unique().sort_values()
aligned = pd.DatetimeIndex(news["aligned_trading_date"])

pos = trading_days.searchsorted(aligned, side="left")  # hari trading pertama yang >= tanggal berita
has_match = pos < len(trading_days)                     # berita setelah hari kurs terakhir tidak punya pasangan
print(f"Artikel dibuang karena melewati hari kurs terakhir: {(~has_match).sum()}")

news = news.loc[has_match].copy()
news["aligned_trading_date"] = trading_days[pos[has_match]]

Artikel dibuang karena melewati hari kurs terakhir: 2


## 3.2 Agregasi harian & merge dengan kurs

In [26]:
news_sorted = news.sort_values("date_raw_dt_wib")

# news_count cuma untuk referensi (menandai hari tanpa berita), bukan fitur model
agg_spec = {"news_count": ("title", "size")}

# VADER: statistik skor judul per hari. Ambang 0.05 adalah batas standar positif/negatif VADER.
if "vader_title" in news_sorted.columns:
    agg_spec.update({
        "vader_title_mean":      ("vader_title", "mean"),
        "vader_title_min":       ("vader_title", "min"),
        "vader_title_max":       ("vader_title", "max"),
        "vader_title_std":       ("vader_title", "std"),
        "vader_title_pos_ratio": ("vader_title", lambda x: (x.dropna() > 0.05).mean()),
        "vader_title_neg_ratio": ("vader_title", lambda x: (x.dropna() < -0.05).mean()),
    })

# LM: rata-rata skor per hari
lm_cols = [c for c in news_sorted.columns if c.startswith(("lm_title_", "lm_text_"))]
agg_spec.update({c: (c, "mean") for c in lm_cols})

daily_news = news_sorted.groupby("aligned_trading_date").agg(**agg_spec).reset_index()

# Hari yang cuma punya 1 berita std-nya NaN, ganti jadi 0
if "vader_title_std" in daily_news.columns:
    daily_news["vader_title_std"] = daily_news["vader_title_std"].fillna(0.0)

daily_news

,aligned_trading_date,news_count,vader_title_mean,vader_title_min,vader_title_max,vader_title_std,vader_title_pos_ratio,vader_title_neg_ratio,lm_title_negative,lm_title_positive,lm_title_uncertainty,lm_title_litigious,lm_title_net,lm_text_negative,lm_text_positive,lm_text_uncertainty,lm_text_litigious,lm_text_net
0,2021-09-06,1,-0.31820,-0.3182,-0.3182,0.000000,0.000000,1.000000,0.076923,0.000000,0.076923,0.0,-0.076923,0.023179,0.003311,0.019868,0.000000,-0.019868
1,2021-09-08,1,-0.34000,-0.3400,-0.3400,0.000000,0.000000,1.000000,0.000000,0.000000,0.090909,0.0,0.000000,0.046174,0.011873,0.014512,0.003958,-0.034301
2,2021-09-09,1,0.07720,0.0772,0.0772,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.026696,0.003337,0.011123,0.003337,-0.023359
3,2021-09-13,3,-0.32460,-0.5719,0.0000,0.293682,0.000000,0.666667,0.046032,0.000000,0.000000,0.0,-0.046032,0.023735,0.008519,0.003971,0.005237,-0.015216
4,2021-09-14,2,-0.20095,-0.4019,0.0000,0.284186,0.000000,0.500000,0.000000,0.000000,0.000000,0.0,0.000000,0.024492,0.004124,0.008669,0.002356,-0.020368
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1071,2026-08-26,3,0.17440,-0.2732,0.7964,0.555718,0.333333,0.333333,0.000000,0.022222,0.000000,0.0,0.022222,0.031929,0.003600,0.012285,0.000747,-0.028329
1072,2026-08-27,1,0.00000,0.0000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.022284,0.008357,0.004178,0.001393,-0.013928
1073,2026-08-28,2,-0.10940,-0.2960,0.0772,0.263892,0.500000,0.500000,0.000000,0.000000,0.000000,0.0,0.000000,0.021471,0.014086,0.014105,0.004818,-0.007385
1074,2026-08-31,10,-0.21439,-0.7506,0.3400,0.386601,0.100000,0.500000,0.051743,0.000000,0.000000,0.0,-0.051743,0.021309,0.004941,0.006305,0.002322,-0.016368


In [27]:
# Samakan tipe kolom tanggal dan buang jamnya
bi["Tanggal_dt"] = pd.to_datetime(bi["Tanggal_dt"]).dt.normalize()
daily_news["aligned_trading_date"] = pd.to_datetime(daily_news["aligned_trading_date"]).dt.normalize()

# Harusnya 0 semua. Kalau ada tanggal dobel, jumlah baris bakal membengkak setelah merge.
print("Tanggal dobel di kurs  :", bi["Tanggal_dt"].duplicated().sum())
print("Tanggal dobel di berita:", daily_news["aligned_trading_date"].duplicated().sum())

merged_data = pd.merge(
    bi,
    daily_news,
    left_on="Tanggal_dt",
    right_on="aligned_trading_date",
    how="left",
    validate="one_to_one",  # langsung error kalau ada tanggal dobel
).drop(columns="aligned_trading_date")

# Hari tanpa berita: news_count = 0 dan skor sentimen dianggap netral (0)
label_cols = [c for c in merged_data.columns if c.startswith(("vader_title_", "lm_title_", "lm_text_"))]
merged_data["news_count"] = merged_data["news_count"].fillna(0).astype(int)
merged_data[label_cols] = merged_data[label_cols].fillna(0.0)

merged_data

Tanggal dobel di kurs  : 0
Tanggal dobel di berita: 0


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam,Tanggal,news_count,vader_title_mean,...,lm_title_negative,lm_title_positive,lm_title_uncertainty,lm_title_litigious,lm_title_net,lm_text_negative,lm_text_positive,lm_text_uncertainty,lm_text_litigious,lm_text_net
0,1202,1,14377.53,14234.47,2021-09-01,09/01/2021,12:00 AM,2021-09-01,0,0.00000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,1201,1,14355.42,14212.58,2021-09-02,09/02/2021,12:00 AM,2021-09-02,0,0.00000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,1200,1,14352.41,14209.60,2021-09-03,09/03/2021,12:00 AM,2021-09-03,0,0.00000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,1199,1,14332.31,14189.70,2021-09-06,09/06/2021,12:00 AM,2021-09-06,1,-0.31820,...,0.076923,0.000000,0.076923,0.0,-0.076923,0.023179,0.003311,0.019868,0.000000,-0.019868
4,1198,1,14310.20,14167.81,2021-09-07,09/07/2021,12:00 AM,2021-09-07,0,0.00000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1197,5,1,17791.51,17614.49,2026-08-26,08/26/2026,12:00 AM,2026-08-26,3,0.17440,...,0.000000,0.022222,0.000000,0.0,0.022222,0.031929,0.003600,0.012285,0.000747,-0.028329
1198,4,1,17805.58,17628.42,2026-08-27,08/27/2026,12:00 AM,2026-08-27,1,0.00000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.022284,0.008357,0.004178,0.001393,-0.013928
1199,3,1,17850.81,17673.19,2026-08-28,08/28/2026,12:00 AM,2026-08-28,2,-0.10940,...,0.000000,0.000000,0.000000,0.0,0.000000,0.021471,0.014086,0.014105,0.004818,-0.007385
1200,2,1,17791.51,17614.49,2026-08-31,08/31/2026,12:00 AM,2026-08-31,10,-0.21439,...,0.051743,0.000000,0.000000,0.0,-0.051743,0.021309,0.004941,0.006305,0.002322,-0.016368


In [28]:
# Ambil kolom yang dibutuhkan saja: kurs (untuk target), news_count (referensi), dan skor sentimen
merged_data_clean = merged_data[["Tanggal_dt", "Kurs Jual", "Kurs Beli", "news_count", *label_cols]].copy()
merged_data_clean

,Tanggal_dt,Kurs Jual,Kurs Beli,news_count,vader_title_mean,vader_title_min,vader_title_max,vader_title_std,vader_title_pos_ratio,vader_title_neg_ratio,lm_title_negative,lm_title_positive,lm_title_uncertainty,lm_title_litigious,lm_title_net,lm_text_negative,lm_text_positive,lm_text_uncertainty,lm_text_litigious,lm_text_net
0,2021-09-01,14377.53,14234.47,0,0.00000,0.0000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2021-09-02,14355.42,14212.58,0,0.00000,0.0000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,2021-09-03,14352.41,14209.60,0,0.00000,0.0000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,2021-09-06,14332.31,14189.70,1,-0.31820,-0.3182,-0.3182,0.000000,0.000000,1.000000,0.076923,0.000000,0.076923,0.0,-0.076923,0.023179,0.003311,0.019868,0.000000,-0.019868
4,2021-09-07,14310.20,14167.81,0,0.00000,0.0000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1197,2026-08-26,17791.51,17614.49,3,0.17440,-0.2732,0.7964,0.555718,0.333333,0.333333,0.000000,0.022222,0.000000,0.0,0.022222,0.031929,0.003600,0.012285,0.000747,-0.028329
1198,2026-08-27,17805.58,17628.42,1,0.00000,0.0000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.022284,0.008357,0.004178,0.001393,-0.013928
1199,2026-08-28,17850.81,17673.19,2,-0.10940,-0.2960,0.0772,0.263892,0.500000,0.500000,0.000000,0.000000,0.000000,0.0,0.000000,0.021471,0.014086,0.014105,0.004818,-0.007385
1200,2026-08-31,17791.51,17614.49,10,-0.21439,-0.7506,0.3400,0.386601,0.100000,0.500000,0.051743,0.000000,0.000000,0.0,-0.051743,0.021309,0.004941,0.006305,0.002322,-0.016368


## 3.3 Target & fitur kurs (sama dengan baseline)
Fitur kurs sama persis dengan baseline: lag 1–5 Kurs Tengah, ditambah rata-rata dan std rolling 5 hari
yang dihitung dari `shift(1)`. Skor sentimen dipakai apa adanya, tanpa lag, rolling, atau diff.

Target adalah log return hari trading berikutnya (t → t+1). Kolom `ret` (log return hari ini) hanya dipakai untuk
baseline naive "Return kemarin", bukan sebagai fitur.


In [29]:
data = merged_data_clean.sort_values("Tanggal_dt", ascending=True).reset_index(drop=True)

# Kurs Tengah, return hari ini (untuk baseline naive), dan target (return hari berikutnya)
data["kurs_mid"] = (data["Kurs Jual"] + data["Kurs Beli"]) / 2
data["ret"] = np.log(data["kurs_mid"]).diff()
data["target"] = data["ret"].shift(-1)

# Fitur kurs, sama dengan baseline. Rolling pakai shift(1) supaya kurs hari ini tidak ikut terhitung.
for lag in range(1, 6):
    data[f"kurs_lag{lag}"] = data["kurs_mid"].shift(lag)
data["kurs_roll_mean5"] = data["kurs_mid"].shift(1).rolling(5).mean()
data["kurs_roll_std5"] = data["kurs_mid"].shift(1).rolling(5).std()

feat_ts = [f"kurs_lag{lag}" for lag in range(1, 6)] + ["kurs_roll_mean5", "kurs_roll_std5"]

# Fitur sentimen, dikelompokkan per metode labelling (metode yang dinonaktifkan otomatis hilang)
LABEL_GROUPS = {
    "VADER":    [c for c in data.columns if c.startswith("vader_")],
    "LM judul": [c for c in data.columns if c.startswith("lm_title_")],
    "LM isi":   [c for c in data.columns if c.startswith("lm_text_")],
}
LABEL_GROUPS = {name: cols for name, cols in LABEL_GROUPS.items() if cols}
feat_label = [c for cols in LABEL_GROUPS.values() for c in cols]

# Buang baris awal (lag/rolling belum lengkap) dan baris terakhir (target kosong) sebelum split
dataset = data.dropna(subset=["ret", "target", *feat_ts, *feat_label]).reset_index(drop=True)

print(dataset.shape)
n_features = {"Kurs (baseline)": len(feat_ts), **{name: len(cols) for name, cols in LABEL_GROUPS.items()}}
print(n_features, "| total fitur:", len(feat_ts) + len(feat_label))
dataset[["Tanggal_dt", "kurs_mid", "ret", "target", *feat_ts[:3]]].head()

(1196, 30)
{'Kurs (baseline)': 7, 'VADER': 6, 'LM judul': 5, 'LM isi': 5} | total fitur: 23


,Tanggal_dt,kurs_mid,ret,target,kurs_lag1,kurs_lag2,kurs_lag3
0,2021-09-08,14195.005,-0.003095,0.004989,14239.005,14261.005,14281.005
1,2021-09-09,14266.000,0.004989,0.000420,14195.005,14239.005,14261.005
2,2021-09-10,14272.000,0.000420,-0.003298,14266.000,14195.005,14239.005
3,2021-09-13,14225.005,-0.003298,0.002457,14272.000,14266.000,14195.005
4,2021-09-14,14260.000,0.002457,-0.000210,14225.005,14272.000,14266.000


# 4. Split Data (70 / 15 / 15, kronologis)

In [30]:
dataset = dataset.sort_values("Tanggal_dt", ascending=True).reset_index(drop=True)

n = len(dataset)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = dataset.iloc[:train_end].copy()
val = dataset.iloc[train_end:val_end].copy()
test = dataset.iloc[val_end:].copy()

# Buang baris terakhir train & val, karena target-nya (t+1) adalah hari pertama set berikutnya
train = train.iloc[:-1].copy()
val = val.iloc[:-1].copy()

assert train["Tanggal_dt"].max() < val["Tanggal_dt"].min() < test["Tanggal_dt"].min()

total = len(train) + len(val) + len(test)
for name, part in [("Train", train), ("Val", val), ("Test", test)]:
    start, end = part["Tanggal_dt"].min().date(), part["Tanggal_dt"].max().date()
    print(f"{name:5}: {len(part):5d} ({len(part) / total:.1%}) | {start} -> {end}")

Train:   836 (70.0%) | 2021-09-08 -> 2025-02-18
Val  :   178 (14.9%) | 2025-02-20 -> 2025-11-20
Test :   180 (15.1%) | 2025-11-24 -> 2026-08-31


# 5. Dataset Final

In [31]:
# Isi dataset final: tanggal, kurs_mid & ret (referensi), fitur kurs, fitur sentimen, dan target
final_cols = ["Tanggal_dt", "kurs_mid", "ret", *feat_ts, *feat_label, "target"]

final_train = train[final_cols].reset_index(drop=True)
final_val = val[final_cols].reset_index(drop=True)
final_test = test[final_cols].reset_index(drop=True)

for name, part in [("Train", final_train), ("Val", final_val), ("Test", final_test)]:
    assert part.isna().sum().sum() == 0, f"{name} masih ada NaN"
    print(f"{name:5}: {part.shape}")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
for name, part in [("train", final_train), ("val", final_val), ("test", final_test)]:
    part.to_csv(PROCESSED_DIR / f"{name}.csv", index=False, date_format="%Y-%m-%d")

Train: (836, 27)
Val  : (178, 27)
Test : (180, 27)


# 5b. Export untuk Tim

Menulis bundle ke `data/export/`: tabel artikel bersih, tabel harga harian dengan kolom `split` terkunci, `splits.json`,
sentimen baseline Tugas 2 (file terpisah), dan data dictionary. Embedding dibuat terpisah (opsional) lewat `src/make_embeddings.py`.


In [32]:
# Data dictionary yang ikut di-export ke data/export/DATA_DICTIONARY.md
DATA_DICTIONARY_MD = r'''# Data Dictionary: bundle export `data/export/`

Dibuat oleh notebook `pipeline` (bagian 5b). Semua tanggal dan jam memakai **WIB (Asia/Jakarta)**.
Format tabel: parquet (kalau `pyarrow` terpasang) atau CSV sebagai cadangan. Baca dengan `pd.read_parquet` / `pd.read_csv`.

## Daftar file

| File | Isi | Kunci |
|---|---|---|
| `articles.parquet` | Artikel bersih, tanpa fitur sentimen | `article_id` |
| `prices_daily.parquet` | Kurs harian, target, fitur harga, dan kolom `split` | `date` |
| `splits.json` | Batas tanggal train/val/test (terkunci) | |
| `sentiment_articles.parquet` | Skor VADER/LM per artikel (baseline Tugas 2) | `article_id` |
| `sentiment_daily.parquet` | Skor sentimen teragregasi per hari trading (baseline Tugas 2) | `date` |
| `embeddings_<tag>.npy` + `.json` | Cache embedding per artikel (dibuat `src/make_embeddings.py`, opsional) | baris ke-i = `article_id` i |

## `articles`

| Kolom | Tipe | Arti |
|---|---|---|
| `article_id` | int | ID urut 0..n-1, diurutkan menurut `published_wib` lalu `url`. Baris ke-i di file embedding = artikel ini. |
| `url` | str | URL artikel (unik). |
| `section` | str | Section CNBC. Bisa kosong. |
| `keyword_matched` | str | Keyword pencarian yang menemukan artikel. Bisa kosong. |
| `title` | str | Judul asli. |
| `preview_summary` | str | Ringkasan dari halaman daftar. |
| `full_text` | str | Isi artikel (minimal 350 kata). |
| `text_clean` | str | Judul (2x) + ringkasan + isi, dibersihkan (tanpa HTML/URL/emoji/tanda baca, huruf kecil). Dipakai untuk skor LM dan bag-of-words. Untuk model transformer sebaiknya pakai `title`/`full_text` asli. |
| `published_wib` | datetime | Waktu terbit dalam WIB. |
| `date` | datetime | Tanggal terbit (WIB), tanpa jam. |
| `jam` | str | Jam terbit WIB, format 24 jam `HH:MM`. |
| `aligned_trading_date` | datetime | Hari trading tempat artikel dipasangkan dengan kurs. Aturan: terbit **setelah 15:00 WIB** dihitung ke hari berikutnya; Sabtu/Minggu digeser ke Senin; lalu digeser ke hari trading terdekat yang ada di data kurs. Selalu ada di `prices_daily`/`sentiment_daily` kalau hari itu bukan hari yang dibuang (lihat catatan). |

## `prices_daily`

Satu baris per hari trading. Hanya baris yang masuk salah satu split.

| Kolom | Arti |
|---|---|
| `date` | Hari trading t. |
| `split` | `train`, `val`, atau `test` (terkunci, jangan dihitung ulang). |
| `kurs_jual`, `kurs_beli` | Kurs transaksi BI (IDR per USD). |
| `kurs_mid` | (`kurs_jual` + `kurs_beli`) / 2. |
| `ret` | Log return hari ini: `ln(kurs_mid[t]) - ln(kurs_mid[t-1])`. **Bukan fitur**, hanya untuk baseline naif "return kemarin". |
| `target` | Log return hari trading berikutnya: `ret[t+1]`. Ini yang diprediksi. |
| `kurs_lag1` ... `kurs_lag5` | `kurs_mid` pada t-1 ... t-5 (level kurs, bukan return). |
| `kurs_roll_mean5` | Rata-rata `kurs_mid` t-1..t-5 (rolling 5 hari, di-`shift(1)` sehingga kurs hari t tidak ikut). |
| `kurs_roll_std5` | Std rolling 5 hari dengan `shift(1)` yang sama. |

Fitur harga = 7 kolom (`kurs_lag1`..`kurs_lag5`, `kurs_roll_mean5`, `kurs_roll_std5`).

## `splits.json`

Split kronologis 70/15/15 dari dataset yang sudah dibersihkan. Baris terakhir train dan val **dibuang**, karena target-nya (t+1) adalah hari pertama split berikutnya. Jadi tidak ada target yang bocor lintas split, dan `split` + tanggal di file ini adalah satu-satunya sumber batas.

## `sentiment_articles` / `sentiment_daily` (baseline Tugas 2)

Disimpan terpisah supaya eksperimen NLP baru tidak otomatis bercampur dengan fitur Tugas 2.

- `vader_title`: skor compound VADER dari judul asli (-1..1).
- `lm_title_*` / `lm_text_*`: proporsi kata Loughran-McDonald (`negative`, `positive`, `uncertainty`, `litigious`) dibanding total kata, dari judul / isi artikel. `*_net` = `positive` - `negative`.
- `sentiment_daily` meng-agregasi per `aligned_trading_date`: `news_count`, `vader_title_{mean,min,max,std,pos_ratio,neg_ratio}`, dan rata-rata tiap skor LM. **Hari tanpa berita: `news_count` = 0 dan semua skor diisi 0.0** (dianggap netral), jadi 0 di hari itu bukan berarti skor benar-benar nol.

## Evaluasi

Pakai `from eval_utils import evaluate` (file `src/eval_utils.py`) untuk semua metrik: MAE, MSE, RMSE, DirAcc, PredStd. `naive_baselines()` memberi pembanding "Prediksi 0" dan "Return kemarin". DirAcc melewati hari dengan target tepat 0.

## Catatan

- Fitur harga adalah **level** kurs, bukan return. Kurs naik dari sekitar 14 ribu (2021) ke sekitar 17,8 ribu (2026), jadi level di val/test kemungkinan keluar dari rentang train; model berbasis pohon tidak bisa ekstrapolasi dan scaler di-fit pada rentang yang berbeda. Kalau mau, tambahkan fitur berbasis return di eksperimen masing-masing, tapi pakai split yang sama.
- Sebagian judul punya kata yang menempel (misalnya `risinggeopoliticalrisks`) akibat scraping. Ini sudah ada di `title` mentah dan ikut ke `text_clean`.
- Embedding hanya valid untuk versi `articles` yang sidik jarinya sama. Cek dengan membandingkan `articles_fingerprint` di file `embeddings_<tag>.json` dengan yang dicetak notebook saat export.
'''


In [33]:
EXPORT_DIR = Path("data/export")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# ---- 1. Tabel artikel bersih (tanpa fitur sentimen) ----
art = news.sort_values(["date_raw_dt_wib", "url"]).reset_index(drop=True)
art["article_id"] = np.arange(len(art), dtype="int64")
art["date"] = art["date_raw_dt_wib"].dt.normalize()
art["jam"] = art["date_raw_dt_wib"].dt.strftime("%H:%M")  # 24 jam, WIB
art = art.rename(columns={"date_raw_dt_wib": "published_wib", "lower_text": "text_clean"})

ARTICLE_COLS = [
    "article_id", "url", "section", "keyword_matched", "title", "preview_summary", "full_text",
    "text_clean", "published_wib", "date", "jam", "aligned_trading_date",
]
articles = art[ARTICLE_COLS].copy()

not_null = [c for c in ARTICLE_COLS if c not in ("section", "keyword_matched")]
assert articles[not_null].notna().all().all(), "ada nilai kosong di kolom wajib"
assert articles["url"].is_unique and articles["article_id"].is_unique
assert articles["aligned_trading_date"].isin(trading_days).all(), "ada artikel yang tidak jatuh di hari trading"

# ---- 2. Tabel harga harian + split terkunci ----
prices = pd.concat(
    [train.assign(split="train"), val.assign(split="val"), test.assign(split="test")],
    ignore_index=True,
).rename(columns={"Tanggal_dt": "date", "Kurs Jual": "kurs_jual", "Kurs Beli": "kurs_beli"})

assert feat_ts == PRICE_FEATURES, "fitur harga beda dengan PRICE_FEATURES di sel fungsi bantu"
PRICE_COLS = ["date", "split", "kurs_jual", "kurs_beli", "kurs_mid", "ret", "target", *feat_ts]
prices = prices[PRICE_COLS]

assert prices[PRICE_COLS].notna().all().all()
assert prices["date"].is_unique and prices["date"].is_monotonic_increasing
assert list(prices["split"].unique()) == ["train", "val", "test"]
for name, part in prices.groupby("split", sort=False):
    # target hari t harus sama dengan ret hari trading berikutnya (di dalam split yang sama)
    assert np.allclose(part["target"].iloc[:-1].to_numpy(), part["ret"].iloc[1:].to_numpy()), name

splits_info = {
    "rule": "kronologis 70/15/15 dari dataset bersih; baris terakhir train & val dibuang karena target (t+1) jatuh di split berikutnya",
    "cutoff_hour_wib": CUTOFF_HOUR,
    "n_rows_before_gap_drop": int(len(dataset)),
    **{
        name: {"start": str(part["date"].min().date()), "end": str(part["date"].max().date()), "n": int(len(part))}
        for name, part in prices.groupby("split", sort=False)
    },
}
(EXPORT_DIR / "splits.json").write_text(json.dumps(splits_info, indent=2), encoding="utf-8")

# ---- 3. Sentimen Tugas 2 (baseline), file terpisah ----
sent_cols = [c for c in art.columns if c == "vader_title" or c.startswith(("lm_title_", "lm_text_"))]
sentiment_articles = art[["article_id", "aligned_trading_date", *sent_cols]].copy()
sentiment_daily = merged_data_clean[["Tanggal_dt", "news_count", *label_cols]].rename(columns={"Tanggal_dt": "date"})

# ---- tulis semua ----
paths = [
    write_table(articles, EXPORT_DIR / "articles"),
    write_table(prices, EXPORT_DIR / "prices_daily"),
    write_table(sentiment_articles, EXPORT_DIR / "sentiment_articles"),
    write_table(sentiment_daily, EXPORT_DIR / "sentiment_daily"),
]

(EXPORT_DIR / "DATA_DICTIONARY.md").write_text(DATA_DICTIONARY_MD, encoding="utf-8")

print(f"articles : {articles.shape} | prices_daily: {prices.shape}")
print(json.dumps({k: v for k, v in splits_info.items() if k in ("train", "val", "test")}, indent=2))
for p in paths:
    print(f"{p.name:32}{p.stat().st_size / 1e6:8.2f} MB")

print("\nSidik jari artikel (untuk cache embedding):", articles_fingerprint(articles))
print("Langkah berikut: python src/make_embeddings.py --model <nama-model> --tag <tag>")


articles : (4332, 12) | prices_daily: (1194, 14)
{
  "train": {
    "start": "2021-09-08",
    "end": "2025-02-18",
    "n": 836
  },
  "val": {
    "start": "2025-02-20",
    "end": "2025-11-20",
    "n": 178
  },
  "test": {
    "start": "2025-11-24",
    "end": "2026-08-31",
    "n": 180
  }
}
articles.parquet                   24.37 MB
prices_daily.parquet                0.11 MB
sentiment_articles.parquet          0.21 MB
sentiment_daily.parquet             0.11 MB

Sidik jari artikel (untuk cache embedding): 39283221d34c8e3455025c04355671964f56036a
Langkah berikut: python src/make_embeddings.py --model <nama-model> --tag <tag>


# 6. Setup Modelling
- Normalisasi pakai `StandardScaler` yang di-fit **di train saja**, lalu dipakai juga untuk val dan test.
- Semua model pakai **parameter default** (tanpa tuning). `random_state` hanya supaya hasilnya bisa diulang.
- Metrik dihitung oleh `src/eval_utils.py` (modul bersama): MAE, MSE, RMSE, dan DirAcc (seberapa sering arah naik/turunnya tertebak benar; hari dengan target tepat 0 tidak dihitung).
- `PredStd` mendekati 0 artinya model menebak angka yang hampir sama terus. Ini wajar di Lasso/ElasticNet/SVR default karena nilai target sangat kecil.


In [34]:
pd.set_option("display.float_format", lambda x: f"{x:.6g}")

# Semua model pakai parameter default
MODELS = {
    "LinearRegression": lambda: LinearRegression(),
    "Ridge":            lambda: Ridge(),
    "Lasso":            lambda: Lasso(),
    "ElasticNet":       lambda: ElasticNet(),
    "BayesianRidge":    lambda: BayesianRidge(),
    "KNN":              lambda: KNeighborsRegressor(),
    "SVR":              lambda: SVR(),
    "DecisionTree":     lambda: DecisionTreeRegressor(random_state=SEED),
    "RandomForest":     lambda: RandomForestRegressor(random_state=SEED, n_jobs=-1),
    "ExtraTrees":       lambda: ExtraTreesRegressor(random_state=SEED, n_jobs=-1),
    "AdaBoost":         lambda: AdaBoostRegressor(random_state=SEED),
    "GradientBoosting": lambda: GradientBoostingRegressor(random_state=SEED),
}

# XGBoost dan LightGBM opsional, dilewati kalau belum terinstall
try:
    from xgboost import XGBRegressor
    MODELS["XGBoost"] = lambda: XGBRegressor(random_state=SEED, n_jobs=-1, verbosity=0)
except ImportError:
    print("xgboost belum terinstall, dilewati")

try:
    from lightgbm import LGBMRegressor
    MODELS["LightGBM"] = lambda: LGBMRegressor(random_state=SEED, n_jobs=-1, verbosity=-1)
except ImportError:
    print("lightgbm belum terinstall, dilewati")

# evaluate() dan METRICS diimpor dari src/eval_utils.py (dipakai bersama seluruh tim)


y_tr, y_va, y_te = final_train["target"], final_val["target"], final_test["target"]


def prepare(cols):
    # Scaler di-fit di train saja, lalu dipakai juga untuk val & test
    scaler = StandardScaler().fit(final_train[cols])
    return (
        scaler.transform(final_train[cols]),
        scaler.transform(final_val[cols]),
        scaler.transform(final_test[cols]),
    )


def fit_eval(make_model, X_tr, X_va, X_te):
    # Latih di train, lalu evaluasi di val & test
    model = make_model().fit(X_tr, y_tr)
    results = {}
    for split, X, y in [("Val", X_va, y_va), ("Test", X_te, y_te)]:
        pred = model.predict(X)
        results[split] = evaluate(y, pred)  # sudah termasuk PredStd
    return results


# Pembanding tanpa model: selalu tebak 0, atau tebak sama dengan return hari ini
eval_splits = [("Val", y_va, final_val), ("Test", y_te, final_test)]
naive = pd.DataFrame(
    [{"Baseline": "Prediksi 0", "Split": s, **evaluate(y, np.zeros(len(y)))} for s, y, _ in eval_splits]
    + [{"Baseline": "Return kemarin", "Split": s, **evaluate(y, part["ret"])} for s, y, part in eval_splits]
)

print(f"{len(MODELS)} model: {list(MODELS)}")
naive

xgboost belum terinstall, dilewati
lightgbm belum terinstall, dilewati
12 model: ['LinearRegression', 'Ridge', 'Lasso', 'ElasticNet', 'BayesianRidge', 'KNN', 'SVR', 'DecisionTree', 'RandomForest', 'ExtraTrees', 'AdaBoost', 'GradientBoosting']


,Baseline,Split,MAE,MSE,RMSE,DirAcc,PredStd
0,Prediksi 0,Val,0.00246991,1.14367e-05,0.00338182,0,0
1,Prediksi 0,Test,0.00220518,8.45542e-06,0.00290782,0,0
2,Return kemarin,Val,0.00346751,2.12584e-05,0.00461068,0.505682,0.00339888
3,Return kemarin,Test,0.00308416,1.58166e-05,0.00397701,0.545455,0.0028874


# 7. Baseline vs Baseline + NLP
- **Baseline**: fitur kurs saja (`kurs_lag1`–`kurs_lag5`, `kurs_roll_mean5`, `kurs_roll_std5`).
- **Baseline + NLP**: fitur kurs ditambah semua fitur sentimen (VADER, LM judul, LM isi).

Semua model dilatih di kedua set fitur, lalu hasilnya dibandingkan.


In [35]:
FEATURE_SETS = {
    "Baseline": feat_ts,
    "Baseline + NLP": feat_ts + feat_label,
}

rows = []
for set_name, cols in FEATURE_SETS.items():
    X_tr, X_va, X_te = prepare(cols)
    for model_name, make_model in MODELS.items():
        res = fit_eval(make_model, X_tr, X_va, X_te)
        for split in ("Val", "Test"):
            rows.append({
                "Model": model_name, "Set fitur": set_name,
                "n_fitur": len(cols), "Split": split, **res[split],
            })
    print(f"selesai: {set_name} ({len(cols)} fitur)")

results = pd.DataFrame(rows)

selesai: Baseline (7 fitur)
selesai: Baseline + NLP (23 fitur)


In [36]:
set_order = list(FEATURE_SETS)

# Ringkasan: rata-rata metrik semua model untuk tiap set fitur
summary_index = pd.MultiIndex.from_product([["Val", "Test"], set_order], names=["Split", "Set fitur"])
summary = results.groupby(["Split", "Set fitur"])[METRICS].mean().reindex(summary_index)
print("Rata-rata metrik semua model per set fitur:")
display(summary)

# Detail per model untuk tiap metrik
for split in ("Val", "Test"):
    for metric in METRICS:
        table = (
            results[results["Split"] == split]
            .pivot(index="Model", columns="Set fitur", values=metric)[set_order]
            .reindex(list(MODELS))
        )
        print(f"\n=== {metric} | {split} ===")
        display(table)

Rata-rata metrik semua model per set fitur:


MAE         MSE       RMSE   DirAcc
Split Set fitur                                                
Val   Baseline       0.00271822 1.31942e-05 0.00361453 0.502841
      Baseline + NLP 0.00276793 1.40005e-05 0.00370298 0.482481
Test  Baseline       0.00245546 1.01733e-05 0.00316755 0.484375
      Baseline + NLP 0.00256792 1.20538e-05 0.00336813 0.494318


=== MAE | Val ===


Set fitur,Baseline,Baseline + NLP
Model,,
LinearRegression,0.00252672,0.00255784
Ridge,0.00251092,0.00254408
Lasso,0.00246464,0.00246464
ElasticNet,0.00246464,0.00246464
BayesianRidge,0.00250102,0.00250695
KNN,0.00284737,0.00276367
SVR,0.00310973,0.00310973
DecisionTree,0.00358013,0.00421101
RandomForest,0.00260867,0.00261337



=== MSE | Val ===


Set fitur,Baseline,Baseline + NLP
Model,,
LinearRegression,1.15704e-05,1.18637e-05
Ridge,1.15099e-05,1.18014e-05
Lasso,1.14207e-05,1.14207e-05
ElasticNet,1.14207e-05,1.14207e-05
BayesianRidge,1.14832e-05,1.16656e-05
KNN,1.34228e-05,1.39801e-05
SVR,1.65624e-05,1.65624e-05
DecisionTree,2.13416e-05,2.88627e-05
RandomForest,1.19681e-05,1.22185e-05


... (8 outputs left out)


=== RMSE | Test ===


Set fitur,Baseline,Baseline + NLP
Model,,
LinearRegression,0.00299457,0.00292131
Ridge,0.00298868,0.00292153
Lasso,0.00289376,0.00289376
ElasticNet,0.00289376,0.00289376
BayesianRidge,0.00298408,0.002939
KNN,0.00298064,0.00337317
SVR,0.00355238,0.00355238
DecisionTree,0.00419307,0.00607671
RandomForest,0.0030043,0.00319259



=== DirAcc | Test ===


Set fitur,Baseline,Baseline + NLP
Model,,
LinearRegression,0.477273,0.556818
Ridge,0.4375,0.522727
Lasso,0.590909,0.590909
ElasticNet,0.590909,0.590909
BayesianRidge,0.414773,0.471591
KNN,0.528409,0.448864
SVR,0.590909,0.590909
DecisionTree,0.426136,0.460227
RandomForest,0.420455,0.414773


In [37]:
# Set fitur yang lebih bagus untuk tiap model, dipilih dari RMSE validation (nilai test hanya dilaporkan)
val_res = results[results["Split"] == "Val"]
test_res = results[results["Split"] == "Test"].set_index(["Model", "Set fitur"])
best_rows = val_res.loc[val_res.groupby("Model")["RMSE"].idxmin()]

best_per_model = (
    best_rows.set_index("Model")[["Set fitur", "n_fitur", "RMSE"]]
    .rename(columns={"RMSE": "Val RMSE"})
)
for metric in METRICS:
    best_per_model[f"Test {metric}"] = [
        test_res.loc[(model, set_name), metric]
        for model, set_name in zip(best_per_model.index, best_per_model["Set fitur"])
    ]
best_per_model = best_per_model.reindex(list(MODELS)).sort_values("Val RMSE")

print("Set fitur terbaik per model:")
display(best_per_model)

Set fitur terbaik per model:


,Set fitur,n_fitur,Val RMSE,Test MAE,Test MSE,Test RMSE,Test DirAcc
Model,,,,,,,
ElasticNet,Baseline,7,0.00337945,0.0021809,8.37383e-06,0.00289376,0.590909
Lasso,Baseline,7,0.00337945,0.0021809,8.37383e-06,0.00289376,0.590909
BayesianRidge,Baseline,7,0.00338869,0.00229396,8.90476e-06,0.00298408,0.414773
Ridge,Baseline,7,0.00339263,0.00229696,8.93223e-06,0.00298868,0.4375
LinearRegression,Baseline,7,0.00340153,0.00230292,8.96746e-06,0.00299457,0.477273
AdaBoost,Baseline,7,0.00344297,0.00234884,9.07439e-06,0.00301237,0.477273
RandomForest,Baseline,7,0.00345949,0.00232296,9.02585e-06,0.0030043,0.420455
ExtraTrees,Baseline,7,0.00348428,0.00230994,8.99494e-06,0.00299916,0.431818
GradientBoosting,Baseline + NLP,23,0.00365419,0.00262643,1.08547e-05,0.00329465,0.420455
